In [1]:
from google.cloud import bigquery
from datetime import datetime, timedelta
import random

In [2]:
project_id = 'pcloud2026'
region = 'europe-west8'
db_id = 'test'
table_id = 'test_table'
client = bigquery.Client.from_service_account_json('../secret.json')

In [3]:
def create_dataset(project_id,dataset_id,location):
    # Construct a BigQuery client object.
    
    dataset_full_id = f'{project_id}.{dataset_id}'
    dataset = bigquery.Dataset(dataset_full_id)
    dataset.location = location
    # Send the dataset to the API for creation, with an explicit timeout.
    # Raises google.api_core.exceptions.Conflict if the Dataset already
    # exists within the project.
    dataset = client.create_dataset(dataset, timeout=30)  # Make an API request.
    print("Created dataset {}.{}".format(client.project, dataset.dataset_id))


create_dataset(project_id,db_id,region)

Created dataset pcloud2026.test


In [6]:
def create_table(project_id,dataset_id,table_id):
    table_full_id = f'{project_id}.{dataset_id}.{table_id}'
    schema = [
        bigquery.SchemaField("sensor", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("lat", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("lon", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("value", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("datetime", "DATETIME", mode="REQUIRED"),
    ]

    table = bigquery.Table(table_full_id, schema=schema)
    table = client.create_table(table)  # Make an API request.
    print(f'Created table {table.project}.{table.dataset_id}.{table.table_id}')


create_table(project_id,db_id,table_id)

Created table pcloud2026.test.test_table


In [8]:
def insert(project_id,dataset_id,table_id):
    table_full_id = f'{project_id}.{dataset_id}.{table_id}'

    dt = datetime.strptime('2021-11-17 17:33:00', '%Y-%m-%d %H:%M:%S')  # YYYY-MM-DD [HH:MM:SS]

    sensors = [
        {'id': 'modena','lat':44.64661935128612, 'lon':10.925714194043392},
        {'id': 'reggio emilia', 'lat': 44.69833568636162, 'lon': 10.63119575772369},
        {'id': 'mantova', 'lat': 45.158563488324816, 'lon': 10.793287903622467}
    ]

    for i in range(100):
        dt += timedelta(minutes=1)
        for s in sensors:
            v = 60 -  0.5*i + random.gauss(0,2)
            if i == 50:
                v += 100
            rows = [{'sensor': s['id'], 'lat': s['lat'], 'lon': s['lon'], 'value': v, 'datetime': dt.strftime('%Y-%m-%d %H:%M:%S')}]
            errors = client.insert_rows_json(table_full_id, rows)  # Make an API request.
            if errors == []:
                # print("New rows have been added.")
                pass
            else:
                print("Encountered errors while inserting rows: {}".format(errors))

insert(project_id,db_id,table_id)

New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows have been added.
New rows hav

In [10]:

def query(project_id,db_id,table):
    query = f"""
    SELECT *
    FROM `{project_id}.{db_id}.{table}`
    WHERE sensor = 'reggio emilia'
    LIMIT 100
    """
    query_job = client.query(query)
    for row in query_job:
        #print(row)
        # Row values can be accessed by field name or index.
        print(f'name={row[0]} datetime={row["datetime"]}')

query(project_id,db_id,table_id)

name=reggio emilia datetime=2021-11-17 17:34:00
name=reggio emilia datetime=2021-11-17 17:35:00
name=reggio emilia datetime=2021-11-17 17:36:00
name=reggio emilia datetime=2021-11-17 17:37:00
name=reggio emilia datetime=2021-11-17 17:38:00
name=reggio emilia datetime=2021-11-17 17:39:00
name=reggio emilia datetime=2021-11-17 17:40:00
name=reggio emilia datetime=2021-11-17 17:41:00
name=reggio emilia datetime=2021-11-17 17:42:00
name=reggio emilia datetime=2021-11-17 17:43:00
name=reggio emilia datetime=2021-11-17 17:44:00
name=reggio emilia datetime=2021-11-17 17:45:00
name=reggio emilia datetime=2021-11-17 17:46:00
name=reggio emilia datetime=2021-11-17 17:47:00
name=reggio emilia datetime=2021-11-17 17:48:00
name=reggio emilia datetime=2021-11-17 17:49:00
name=reggio emilia datetime=2021-11-17 17:50:00
name=reggio emilia datetime=2021-11-17 17:51:00
name=reggio emilia datetime=2021-11-17 17:52:00
name=reggio emilia datetime=2021-11-17 17:53:00
name=reggio emilia datetime=2021-11-17 1

In [4]:
querys = f"""
    SELECT *
    FROM `{project_id}.{db_id}.{table_id}`
    WHERE sensor = 'reggio emilia'
    LIMIT 100
    """

df = client.query(querys).to_dataframe()
df.head()

,sensor,lat,lon,value,datetime
0,reggio emilia,44.698336,10.631196,57.830756,2021-11-17 17:34:00
1,reggio emilia,44.698336,10.631196,62.129619,2021-11-17 17:35:00
2,reggio emilia,44.698336,10.631196,55.930631,2021-11-17 17:36:00
3,reggio emilia,44.698336,10.631196,56.971911,2021-11-17 17:37:00
4,reggio emilia,44.698336,10.631196,61.858947,2021-11-17 17:38:00
